In [1]:
from pyscf import gto,scf,mcscf,mrpt
from mrh.my_pyscf import mcpdft
import numpy as np
import pandas as pd
#Uses code from https://github.com/dking072/dsk.git
from dsk.pickle import load_pkl

class ASOrb:
    def __init__(self,pkl_fn):
        data = load_pkl(pkl_fn)
        mol = gto.Mole()
        mol.atom = data["metadata"]["geom"]
        mol.basis = data["metadata"]["basis"]
        mol.charge = data["metadata"]["charge"]
        mol.spin = data["metadata"]["hf_spin"]
        mol.symmetry = True
        mol.unit = data["metadata"]["unit"]
        mol.build()
        self.mol = mol
        nactorbs = data["metadata"]["ncas"]
        nactel = data["metadata"]["nelecas"]

        mf = scf.ROHF(mol)
        mf.mo_coeff = data["mo_coeff"]
        self.mf = mf
    
        ss_to_maxm = {
            0:0,
            0.75:1,
            2:2,
        }

        #Have to adjust AS electrons to max Ms
        i = 0
        ss = np.round(data["energies"].iloc[i]["ss"],2)
        maxm = ss_to_maxm[ss]
        tot_nactel = np.sum(nactel)
        nalpha = tot_nactel//2 + tot_nactel%2
        nbeta = tot_nactel//2
        nactel = (nalpha,nbeta)
        while nalpha - nbeta != maxm:
            nalpha += 1
            nbeta -= 1

        otfnal = data["metadata"]["otfnal"]
        grids_level = data["metadata"]["grid_level"]
        mc = mcpdft.CASCI(mf,otfnal,nactorbs,nactel,grids_level=grids_level)
        mc.mo_coeff = data["mo_coeff"]
        mc.ci = data["ci"][i]
        self.mc = mc
        self.data = data

    def count_core_electrons(self):
        mol = self.mol
        core_electrons = 0
        for atom_Z in mol.atom_charges():
            if atom_Z > 2:
                core_electrons += 2  # 1s² electrons are considered core
        return core_electrons

    def calc_entropies(self):
        from dsk.mrdiagnostics.ss import calc_n_dist, calc_entropies
        return calc_entropies(self.mc)

    def prep_mos(self):
        from deeporb.data_gen import OrbExtract
        self.orbextract = OrbExtract(mol=self.mol,mo_coeff=self.mc.mo_coeff)

    def get_mo(self,i):
        return self.orbextract.extract_nlm(i)

#Example object
pkl_fn = "../../AS_ORB_DATA/DZ1212/acetone-x1.pkl"
obj = ASOrb(pkl_fn)

/tmp/ipykernel_1210235/1107152752.py:2: FutureWarning: Most MC-PDFT and MC-DCFT modules have been moved to pyscf-forge (github.com/pyscf/pyscf-forge) and will be removed from mrh soon.
  from mrh.my_pyscf import mcpdft


## Aug(12,12)

In [2]:
import h5py

class H5File:
    def __init__(self,h5_fn):
        self.h5_fn = h5_fn
        self.onum = 0

    def save_dct(self,dct):
        with h5py.File(self.h5_fn, "a") as f:
            for k,v in dct.items():
                if isinstance(v,dict):
                    for k2,v2 in v.items():
                        f.create_dataset(f"o{self.onum}/{k}_{k2}", data=v2)
                else:
                    f.create_dataset(f"o{self.onum}/{k}", data=v)
        self.onum += 1

import pandas as pd
import numpy as np
#Data can be found in the Zenodo repo https://doi.org/10.5281/zenodo.6644169
alldf = pd.read_excel("../../AS_ORB_DATA/Results/aug_ccpvtz_results.xlsx")
df_dz = pd.read_excel("../../AS_ORB_DATA/Results/Aug1212.xlsx")
alldf["typ"] = [s.split("-")[0] for s in alldf["Name"]]
alldf["good"] = np.abs(df_dz["tPBE0"] - alldf["TBE"]) < 0.55
idx = [i for i in alldf.index if "x1" in alldf["Name"][i]]
good_idx = alldf.iloc[idx,:].query("good")["Name"].tolist()

data_dir = "../../AS_ORB_DATA/Aug1212/"
h5_fn = "../data/as_orbs/aug1212_asonly.h5"
import os
if os.path.isfile(h5_fn):
    os.system(f"rm {h5_fn}")
h5obj = H5File(h5_fn)

from tqdm import tqdm
for name in tqdm(good_idx):
    pkl_fn = f"{data_dir}/{name}.pkl"
    obj = ASOrb(pkl_fn)
    try:
        entropies = obj.calc_entropies()
    #Some systems have occupation probabilities too close to zero which causes a bug
    #Here we just skip the systems (only 2 are skipped)
    except AssertionError:
        print(name,"skipped!")
        continue
    obj.prep_mos()
    # for i in range(len(entropies)):
    for i in range(obj.mc.ncore,obj.mc.ncore+obj.mc.ncas):
        dct = obj.get_mo(i)
        dct["entropy"] = entropies[i]
        h5obj.save_dct(dct)
    # break

 20%|████████████████████▉                                                                                     | 30/152 [00:27<01:52,  1.09it/s]

ccl2-x1 skipped!


 56%|███████████████████████████████████████████████████████████▎                                              | 85/152 [01:19<01:05,  1.02it/s]

octatetraene-x1 skipped!


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 152/152 [02:15<00:00,  1.12it/s]


In [ ]:
#Data can be found in the Zenodo repo https://doi.org/10.5281/zenodo.6644169
alldf = pd.read_excel("../../AS_ORB_DATA/Results/aug_ccpvtz_results.xlsx")
df_dz = pd.read_excel("../../AS_ORB_DATA/Results/DZ1212.xlsx")
alldf["typ"] = [s.split("-")[0] for s in alldf["Name"]]
alldf["good"] = np.abs(df_dz["tPBE0"] - alldf["TBE"]) < 0.55
idx = [i for i in alldf.index if "x1" in alldf["Name"][i]]
good_idx = alldf.iloc[idx,:].query("good")["Name"].tolist()

data_dir = "../../AS_ORB_DATA/DZ1212/"
h5_fn = "../data/dz1212_asonly.h5"
import os
if os.path.isfile(h5_fn):
    os.system(f"rm {h5_fn}")
h5obj = H5File(h5_fn)

from tqdm import tqdm
for name in tqdm(good_idx):
    pkl_fn = f"{data_dir}/{name}.pkl"
    obj = ASOrb(pkl_fn)
    try:
        entropies = obj.calc_entropies()
    #Some systems have occupation probabilities too close to zero which causes a bug
    #Here we just skip the systems (only 2 are skipped)
    except AssertionError:
        print(name,"skipped!")
        continue
    obj.prep_mos()
    # for i in range(len(entropies)):
    for i in range(obj.mc.ncore,obj.mc.ncore+obj.mc.ncas):
        dct = obj.get_mo(i)
        dct["entropy"] = entropies[i]
        h5obj.save_dct(dct)
    # break